In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model

In [2]:
train_dir = "../preprocessing/train"
val_dir = "../preprocessing/validation"
test_dir = "../preprocessing/test"

os.makedirs("../results", exist_ok=True)

In [3]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

for layer in base_model.layers:
    layer.trainable = False

embedding_layer = Dense(
    512,
    activation="relu",
    name="iris_embedding"
)(base_model.output)

model = Model(
    inputs=base_model.input,
    outputs=embedding_layer
)

print(model.output_shape)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 8s 0us/step
(None, 512)


In [4]:
sample_image = os.path.join(
    train_dir,
    os.listdir(train_dir)[0]
)

img = load_img(
    sample_image,
    target_size=(224,224)
)

img_array = img_to_array(img)

img_array = np.expand_dims(
    img_array,
    axis=0
)

img_array = preprocess_input(
    img_array
)

embedding = model.predict(img_array)

print("Embedding Shape:", embedding.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Embedding Shape: (1, 512)


In [5]:
def extract_embedding(image_path):

    img = load_img(
        image_path,
        target_size=(224,224)
    )

    img_array = img_to_array(img)

    img_array = np.expand_dims(
        img_array,
        axis=0
    )

    img_array = preprocess_input(
        img_array
    )

    embedding = model.predict(
        img_array,
        verbose=0
    )

    return embedding.flatten()

In [6]:
train_embeddings = []

train_files = [
    f for f in os.listdir(train_dir)
    if f.endswith(".jpg")
]

for file in tqdm(train_files):

    path = os.path.join(
        train_dir,
        file
    )

    embedding = extract_embedding(path)

    row = [file]

    row.extend(
        embedding.tolist()
    )

    train_embeddings.append(row)

100%|██████████| 14446/14446 [1:59:45<00:00,  2.01it/s]   


In [9]:
import os

print(os.listdir("../results"))

['week1_results', 'week2_results']


In [10]:
len(train_embeddings)

14446

In [11]:
columns = ["Image"]

for i in range(512):
    columns.append(f"F{i}")

train_df = pd.DataFrame(
    train_embeddings,
    columns=columns
)

train_df.to_csv(
    "../results/train_embeddings.csv",
    index=False
)

print(train_df.shape)

(14446, 513)


In [12]:
import os

print(os.path.exists("../results/train_embeddings.csv"))

True


In [13]:
import os

size_mb = os.path.getsize("../results/train_embeddings.csv")/(1024*1024)

print(f"{size_mb:.2f} MB")

78.58 MB


In [14]:
val_embeddings = []

val_files = [
    f for f in os.listdir(val_dir)
    if f.endswith(".jpg")
]

for file in tqdm(val_files):

    path = os.path.join(val_dir, file)

    embedding = extract_embedding(path)

    row = [file]
    row.extend(embedding.tolist())

    val_embeddings.append(row)

100%|██████████| 2549/2549 [07:52<00:00,  5.40it/s]


In [15]:
columns = ["Image"]

for i in range(512):
    columns.append(f"F{i}")

val_df = pd.DataFrame(
    val_embeddings,
    columns=columns
)

val_df.to_csv(
    "../results/validation_embeddings.csv",
    index=False
)

print(val_df.shape)

(2549, 513)


In [16]:
test_embeddings = []

test_files = [
    f for f in os.listdir(test_dir)
    if f.endswith(".jpg")
]

for file in tqdm(test_files):

    path = os.path.join(test_dir, file)

    embedding = extract_embedding(path)

    row = [file]
    row.extend(embedding.tolist())

    test_embeddings.append(row)

100%|██████████| 3000/3000 [10:41<00:00,  4.68it/s]


In [17]:
test_df = pd.DataFrame(
    test_embeddings,
    columns=columns
)

test_df.to_csv(
    "../results/test_embeddings.csv",
    index=False
)

print(test_df.shape)

(3000, 513)


In [18]:
import os

print(os.listdir("../results"))

['test_embeddings.csv', 'train_embeddings.csv', 'validation_embeddings.csv', 'week1_results', 'week2_results']


In [20]:
train_df.to_csv(
    "../results/week3_results/train_embeddings.csv",
    index=False
)

In [21]:
test_df.to_csv(
    "../results/week3_results/test_embeddings.csv",
    index=False
)

In [22]:
val_df.to_csv(
    "../results/week3_results/validation_embeddings.csv",
    index=False
)

In [23]:
train_df.head(100).to_csv(
    "../results/week3_results/sample_train_embeddings.csv",
    index=False
)

In [24]:
test_df.head(100).to_csv(
    "../results/week3_results/sample_test_embeddings.csv",
    index=False
)

In [26]:
val_df.head(100).to_csv(
    "../results/week3_results/sample_validation_embeddings.csv",
    index=False
)

In [27]:
import os

os.makedirs("../models", exist_ok=True)

model.save("../models/resnet50_feature_extractor.keras")